# Attention-Gated TDM Exploration

This notebook tests whether FLUX edit+part token attention can refine Follow-Your-Shape trajectory divergence maps for part-level localization.

The notebook is intentionally localization-only: it does not rerun image generation and does not overwrite existing FYS, attention, LPIPS, or manual-review results.


## Target

Run this notebook top-to-bottom after both existing result folders are present:

- `core/results/follow_your_shape/<case_uid>/seed_<seed>/tdm/smoothed_soft_tdm.npy`
- `core/results/flux_attention_baseline/<case_uid>/seed_<seed>/attention_proxy_smoothed.npy`
- `core/data/partedit_subset/cases/<case_uid>/gt_mask.png`

It builds a fixed hybrid localization map:

```text
tdm_norm = normalize(FYS soft TDM)
attn_norm = normalize(FLUX edit+part token attention)
hybrid_raw = tdm_norm * attn_norm
hybrid_soft = GaussianSmooth(hybrid_raw, sigma=0.7)
hybrid_binary = Otsu(hybrid_soft)
```

So the final hybrid mask is not pure attention. The available attention signal is edit+part token attention, so it can encode both the requested new attribute/object and the target part. It acts as a multiplicative gate over TDM: a region remains strong only when both the trajectory-divergence signal and the edit+part attention signal are high. This can make the visualization look attention-dominated, but low-TDM regions are still suppressed.

The output is a diagnostic comparison of FYS-TDM, FLUX edit+part token attention, and attention-gated TDM. If the hybrid improves localization without collapsing the mask, it becomes the candidate control signal for the next actual editing run.


## 1. Setup

The analysis uses only local files already produced by the controlled revision run. Paths are repo-relative so the notebook can be rerun after cloning the repository and unpacking the artifact bundle.


In [ ]:
from __future__ import annotations

import math
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

NOTEBOOK_PATH = Path.cwd()
REPO_ROOT = NOTEBOOK_PATH
while not (REPO_ROOT / 'pyproject.toml').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('Could not find repository root from current working directory')
    REPO_ROOT = REPO_ROOT.parent

BASE_RESULTS_DIR = REPO_ROOT / 'core' / 'results'
CONTROLLED_REVISION_DIR = BASE_RESULTS_DIR / 'controlled_revision'
RESULT_DIR = BASE_RESULTS_DIR / 'attention_gated_tdm'
FIGURE_DIR = RESULT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FYS_METRICS_PATH = CONTROLLED_REVISION_DIR / 'fys_run_metrics.csv'
ATTENTION_VARIANT = 'part_edit'
ATTENTION_ROOT = BASE_RESULTS_DIR / 'flux_attention_baseline'
ATTENTION_METRICS_PATH = CONTROLLED_REVISION_DIR / 'flux_attention_metrics.csv'
HYBRID_METRICS_PATH = RESULT_DIR / 'hybrid_localization_metrics.csv'
HYBRID_SUMMARY_PATH = RESULT_DIR / 'hybrid_localization_summary.csv'

print('repo:', REPO_ROOT)
print('controlled revision input dir:', CONTROLLED_REVISION_DIR)
print('attention-gated TDM output dir:', RESULT_DIR)


## 2. Define The Hybrid Rule

The rule is fixed for every case and seed. This avoids case-specific tuning.

- `FYS-TDM` supplies trajectory-change evidence: where source-prompt and target-prompt denoising trajectories diverge.
- `FLUX edit+part token attention` supplies token-aware semantic localization: where image tokens attend to the selected edit and part words in the target prompt.
- `Attention-gated TDM` multiplies the two normalized maps, then smooths and thresholds the product.

This means attention does not replace TDM. It only gates TDM. The hybrid is high only when both maps agree; if attention is broad but TDM is weak, the hybrid is weak, and if TDM is broad but attention is focused, the hybrid is narrowed.


In [ ]:
SMOOTHING_SIGMA = 0.7
EPS = 1e-8

def minmax_normalize(array: np.ndarray) -> np.ndarray:
    values = np.asarray(array, dtype=np.float32)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values, dtype=np.float32)
    min_value = float(np.nanmin(values[finite]))
    max_value = float(np.nanmax(values[finite]))
    if max_value - min_value < EPS:
        return np.zeros_like(values, dtype=np.float32)
    return ((values - min_value) / (max_value - min_value)).clip(0, 1).astype(np.float32)


def load_soft_map(path: str) -> np.ndarray:
    return np.asarray(np.load(REPO_ROOT / path), dtype=np.float32)


def load_gt_mask(path: str, map_shape: tuple[int, int]) -> np.ndarray:
    mask_image = Image.open(REPO_ROOT / path).convert('L')
    mask_image = mask_image.resize((map_shape[1], map_shape[0]), resample=Image.Resampling.NEAREST)
    return (np.asarray(mask_image) > 0).astype(bool)


def otsu_threshold(score: np.ndarray, bins: int = 256) -> float:
    values = np.asarray(score, dtype=np.float32).ravel()
    values = values[np.isfinite(values)]
    if values.size == 0 or np.nanmax(values) <= np.nanmin(values):
        return float(np.nanmean(values)) if values.size else 0.0
    hist, edges = np.histogram(values, bins=bins, range=(float(values.min()), float(values.max())))
    hist = hist.astype(np.float64)
    centers = (edges[:-1] + edges[1:]) / 2.0
    weight_total = hist.sum()
    weight_background = np.cumsum(hist)
    weight_foreground = weight_total - weight_background
    mean_background = np.cumsum(hist * centers) / np.maximum(weight_background, EPS)
    mean_foreground = (np.cumsum((hist * centers)[::-1]) / np.maximum(np.cumsum(hist[::-1]), EPS))[::-1]
    variance_between = weight_background * weight_foreground * (mean_background - mean_foreground) ** 2
    variance_between[(weight_background <= 0) | (weight_foreground <= 0)] = -1
    return float(centers[int(np.argmax(variance_between))])


def otsu_binary(score: np.ndarray) -> tuple[np.ndarray, float]:
    threshold = otsu_threshold(score)
    return score > threshold, threshold


def average_precision_from_scores(gt: np.ndarray, score: np.ndarray) -> float:
    labels = gt.astype(bool).ravel()
    values = np.asarray(score, dtype=np.float32).ravel()
    positives = int(labels.sum())
    if positives == 0:
        return float('nan')
    order = np.argsort(-values, kind='mergesort')
    sorted_labels = labels[order]
    true_positives = np.cumsum(sorted_labels)
    ranks = np.arange(1, len(sorted_labels) + 1)
    precision = true_positives / ranks
    return float(precision[sorted_labels].sum() / positives)


def localization_metrics(gt: np.ndarray, soft_score: np.ndarray, binary_mask: np.ndarray) -> dict[str, float]:
    gt = gt.astype(bool)
    binary_mask = binary_mask.astype(bool)
    intersection = np.logical_and(gt, binary_mask).sum()
    union = np.logical_or(gt, binary_mask).sum()
    pred_area = float(binary_mask.mean())
    gt_area = float(gt.mean())
    score_sum = float(np.asarray(soft_score, dtype=np.float64).sum())
    inside_score = float(np.asarray(soft_score, dtype=np.float64)[gt].sum()) if gt.any() else float('nan')
    return {
        'binary_iou': float(intersection / union) if union else float('nan'),
        'soft_ap': average_precision_from_scores(gt, soft_score),
        'pred_area': pred_area,
        'gt_area': gt_area,
        'pred_to_gt_area_ratio': float(pred_area / gt_area) if gt_area > 0 else float('nan'),
        'soft_inside_gt_mass': float(inside_score / score_sum) if score_sum > EPS else float('nan'),
    }

print('Hybrid rule: minmax(TDM) * minmax(attention), gaussian sigma =', SMOOTHING_SIGMA, ', threshold = Otsu')


## 3. Load And Align Inputs

Each row corresponds to one fixed case and seed. We merge FYS-TDM metadata with the FLUX edit+part token attention baseline by `run_uid`, then check that all required arrays and masks exist.


In [ ]:
fys = pd.read_csv(FYS_METRICS_PATH)


def build_attention_metrics_from_root(fys_table: pd.DataFrame, attention_root: Path) -> pd.DataFrame:
    rows = []
    for _, row in fys_table.iterrows():
        run_dir = attention_root / row['case_uid'] / f"seed_{int(row['seed']):03d}"
        raw_path = run_dir / 'attention_proxy_raw.npy'
        smoothed_path = run_dir / 'attention_proxy_smoothed.npy'
        binary_path = run_dir / 'attention_proxy_binary.npy'
        config_path = run_dir / 'run_config.json'
        if not (smoothed_path.exists() and binary_path.exists() and config_path.exists()):
            continue
        smoothed = np.asarray(np.load(smoothed_path), dtype=np.float32)
        binary = np.asarray(np.load(binary_path)).astype(bool)
        gt = load_gt_mask(row['gt_mask'], smoothed.shape)
        metrics = localization_metrics(gt, smoothed, binary)
        config = json.loads(config_path.read_text())
        rows.append({
            'run_uid': row['run_uid'],
            'case_uid': row['case_uid'],
            'seed': int(row['seed']),
            'part_size': row['part_size'],
            'part': row['part'],
            'edit': row['edit'],
            'source_prompt': config.get('source_prompt'),
            'target_prompt': config.get('target_prompt'),
            'gt_area': metrics['gt_area'],
            'pred_attention_area': metrics['pred_area'],
            'pred_to_gt_area_ratio': metrics['pred_to_gt_area_ratio'],
            'binary_iou': metrics['binary_iou'],
            'soft_ap': metrics['soft_ap'],
            'soft_inside_gt_mass': metrics['soft_inside_gt_mass'],
            'attention_raw_path': raw_path.relative_to(REPO_ROOT).as_posix() if raw_path.exists() else '',
            'attention_smoothed_path': smoothed_path.relative_to(REPO_ROOT).as_posix(),
            'attention_binary_path': binary_path.relative_to(REPO_ROOT).as_posix(),
            'attention_config_path': config_path.relative_to(REPO_ROOT).as_posix(),
            'attention_log_path': (run_dir / 'run.log').relative_to(REPO_ROOT).as_posix(),
            'token_mode': config.get('token_mode', ATTENTION_VARIANT),
            'token_indices': config.get('token_indices'),
            'recorded_step_indices': config.get('recorded_step_indices'),
            'layer_ids': config.get('layer_ids'),
        })
    return pd.DataFrame(rows)

if ATTENTION_METRICS_PATH.exists():
    attention = pd.read_csv(ATTENTION_METRICS_PATH)
    # Older edit+part attention tables were generated before token_mode was
    # recorded. Backfill it instead of requiring an expensive rerun.
    if 'token_mode' not in attention.columns:
        attention['token_mode'] = ATTENTION_VARIANT
else:
    attention = build_attention_metrics_from_root(fys, ATTENTION_ROOT)
    if attention.empty:
        raise FileNotFoundError(
            f"Edit+part attention outputs are missing under {ATTENTION_ROOT}. "
            "Run: python core/scripts/run_flux_attention_baseline.py --manifest core/data/partedit_subset/pilot_12_manifest.json --seeds 0,1,2 --token-mode part_edit --execute"
        )
    attention.to_csv(ATTENTION_METRICS_PATH, index=False)
    print('saved:', ATTENTION_METRICS_PATH)

required_fys = ['run_uid', 'case_uid', 'seed', 'part_size', 'part', 'edit', 'gt_mask', 'source_image', 'fys_image', 'smoothed_tdm_path', 'binary_tdm_path']
required_attention = ['run_uid', 'attention_smoothed_path', 'attention_binary_path', 'token_mode', 'token_indices', 'recorded_step_indices', 'layer_ids']
missing_fys_cols = sorted(set(required_fys) - set(fys.columns))
missing_attention_cols = sorted(set(required_attention) - set(attention.columns))
if missing_fys_cols or missing_attention_cols:
    raise KeyError({'missing_fys_cols': missing_fys_cols, 'missing_attention_cols': missing_attention_cols})

runs = fys[required_fys + ['binary_iou', 'soft_ap', 'pred_to_gt_area_ratio', 'soft_inside_gt_mass']].merge(
    attention[required_attention + ['binary_iou', 'soft_ap', 'pred_to_gt_area_ratio', 'soft_inside_gt_mass']],
    on='run_uid',
    suffixes=('_fys', '_attention'),
)

for column in ['gt_mask', 'source_image', 'fys_image', 'smoothed_tdm_path', 'binary_tdm_path', 'attention_smoothed_path', 'attention_binary_path']:
    runs[f'{column}_exists'] = runs[column].map(lambda value: (REPO_ROOT / value).exists())

missing_files = runs[[c for c in runs.columns if c.endswith('_exists')]].eq(False).sum()
print('runs:', runs.shape[0])
print('missing files by field:')
print(missing_files[missing_files > 0])

if missing_files.sum() > 0:
    bad = runs.loc[runs[[c for c in runs.columns if c.endswith('_exists')]].eq(False).any(axis=1), ['run_uid'] + [c for c in runs.columns if c.endswith('_exists')]]
    display(bad)
    raise FileNotFoundError('Some required FYS/attention/mask files are missing.')

runs[['run_uid', 'case_uid', 'seed', 'part_size', 'part', 'edit', 'token_indices', 'recorded_step_indices', 'layer_ids']].head()


## 4. Compute Hybrid Maps And Metrics

This cell writes new hybrid arrays under `core/results/attention_gated_tdm/hybrid_masks/`. It does not modify original FYS or attention outputs.


In [ ]:
HYBRID_MASK_DIR = RESULT_DIR / 'hybrid_masks'
HYBRID_MASK_DIR.mkdir(parents=True, exist_ok=True)

hybrid_rows = []
for _, row in runs.iterrows():
    tdm = load_soft_map(row['smoothed_tdm_path'])
    attention_map = load_soft_map(row['attention_smoothed_path'])
    if tdm.shape != attention_map.shape:
        raise ValueError(f"shape mismatch for {row['run_uid']}: TDM={tdm.shape}, attention={attention_map.shape}")

    gt = load_gt_mask(row['gt_mask'], tdm.shape)
    tdm_norm = minmax_normalize(tdm)
    attention_norm = minmax_normalize(attention_map)
    hybrid_raw = tdm_norm * attention_norm
    hybrid_smooth = gaussian_filter(hybrid_raw, sigma=SMOOTHING_SIGMA).astype(np.float32)
    hybrid_binary, threshold = otsu_binary(hybrid_smooth)

    out_dir = HYBRID_MASK_DIR / row['case_uid'] / f"seed_{int(row['seed']):03d}"
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_path = out_dir / 'attention_gated_tdm_raw.npy'
    smoothed_path = out_dir / 'attention_gated_tdm_smoothed.npy'
    binary_path = out_dir / 'attention_gated_tdm_binary.npy'
    np.save(raw_path, hybrid_raw.astype(np.float32))
    np.save(smoothed_path, hybrid_smooth.astype(np.float32))
    np.save(binary_path, hybrid_binary.astype(np.uint8))

    metrics = localization_metrics(gt, hybrid_smooth, hybrid_binary)
    hybrid_rows.append({
        'run_uid': row['run_uid'],
        'case_uid': row['case_uid'],
        'seed': int(row['seed']),
        'part_size': row['part_size'],
        'part': row['part'],
        'edit': row['edit'],
        'source_image': row['source_image'],
        'gt_mask': row['gt_mask'],
        'fys_image': row['fys_image'],
        'fys_tdm_path': row['smoothed_tdm_path'],
        'attention_path': row['attention_smoothed_path'],
        'hybrid_raw_path': raw_path.relative_to(REPO_ROOT).as_posix(),
        'hybrid_smoothed_path': smoothed_path.relative_to(REPO_ROOT).as_posix(),
        'hybrid_binary_path': binary_path.relative_to(REPO_ROOT).as_posix(),
        'hybrid_threshold': threshold,
        **{f'hybrid_{key}': value for key, value in metrics.items()},
    })

hybrid_metrics = pd.DataFrame(hybrid_rows)
hybrid_metrics.to_csv(HYBRID_METRICS_PATH, index=False)
print('saved:', HYBRID_METRICS_PATH)
hybrid_metrics.head()


## 5. Compare Localization Metrics

The comparison keeps the same 36 runs and reports method-level summaries by part-size bucket.


In [ ]:
comparison_rows = []
for _, row in runs.iterrows():
    base = {
        'run_uid': row['run_uid'],
        'case_uid': row['case_uid'],
        'seed': int(row['seed']),
        'part_size': row['part_size'],
        'part': row['part'],
        'edit': row['edit'],
    }
    comparison_rows.append({
        **base,
        'method': 'FYS-TDM',
        'binary_iou': row['binary_iou_fys'],
        'soft_ap': row['soft_ap_fys'],
        'pred_to_gt_area_ratio': row['pred_to_gt_area_ratio_fys'],
        'soft_inside_gt_mass': row['soft_inside_gt_mass_fys'],
    })
    comparison_rows.append({
        **base,
        'method': 'FLUX edit+part attention',
        'binary_iou': row['binary_iou_attention'],
        'soft_ap': row['soft_ap_attention'],
        'pred_to_gt_area_ratio': row['pred_to_gt_area_ratio_attention'],
        'soft_inside_gt_mass': row['soft_inside_gt_mass_attention'],
    })

for _, row in hybrid_metrics.iterrows():
    comparison_rows.append({
        'run_uid': row['run_uid'],
        'case_uid': row['case_uid'],
        'seed': int(row['seed']),
        'part_size': row['part_size'],
        'part': row['part'],
        'edit': row['edit'],
        'method': 'Attention-gated TDM',
        'binary_iou': row['hybrid_binary_iou'],
        'soft_ap': row['hybrid_soft_ap'],
        'pred_to_gt_area_ratio': row['hybrid_pred_to_gt_area_ratio'],
        'soft_inside_gt_mass': row['hybrid_soft_inside_gt_mass'],
    })

comparison = pd.DataFrame(comparison_rows)
comparison_path = RESULT_DIR / 'hybrid_localization_comparison_long.csv'
comparison.to_csv(comparison_path, index=False)
print('saved:', comparison_path)

def mean_std(series: pd.Series) -> str:
    return f"{series.mean():.3f} ± {series.std(ddof=0):.3f}"

summary = (
    comparison
    .groupby(['method', 'part_size'], as_index=False)
    .agg(
        n_runs=('run_uid', 'count'),
        binary_iou_mean=('binary_iou', 'mean'),
        binary_iou_std=('binary_iou', lambda x: x.std(ddof=0)),
        soft_ap_mean=('soft_ap', 'mean'),
        soft_ap_std=('soft_ap', lambda x: x.std(ddof=0)),
        area_ratio_mean=('pred_to_gt_area_ratio', 'mean'),
        area_ratio_std=('pred_to_gt_area_ratio', lambda x: x.std(ddof=0)),
        inside_mass_mean=('soft_inside_gt_mass', 'mean'),
        inside_mass_std=('soft_inside_gt_mass', lambda x: x.std(ddof=0)),
    )
)
summary.to_csv(HYBRID_SUMMARY_PATH, index=False)
print('saved:', HYBRID_SUMMARY_PATH)

summary_display = summary.copy()
for metric in ['binary_iou', 'soft_ap', 'area_ratio', 'inside_mass']:
    summary_display[metric] = summary_display.apply(lambda r: f"{r[f'{metric}_mean']:.3f} ± {r[f'{metric}_std']:.3f}", axis=1)
summary_display[['method', 'part_size', 'n_runs', 'binary_iou', 'soft_ap', 'area_ratio', 'inside_mass']]


## 6. Visual Metric Check

These charts are diagnostic. A useful hybrid should increase IoU/AP and reduce extreme predicted/GT area ratios, especially for small parts.


In [ ]:
method_order = ['FYS-TDM', 'FLUX edit+part attention', 'Attention-gated TDM']
part_order = ['large', 'medium', 'small']
colors = {'FYS-TDM': '#7c3aed', 'FLUX edit+part attention': '#0284c7', 'Attention-gated TDM': '#16a34a'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=130)
metrics_to_plot = [
    ('binary_iou', 'Binary IoU', 'higher is better'),
    ('soft_ap', 'Soft AP', 'higher is better'),
    ('pred_to_gt_area_ratio', 'Predicted / GT Area', 'closer to 1 is better'),
]

for ax, (metric, title, subtitle) in zip(axes, metrics_to_plot):
    positions = np.arange(len(part_order))
    width = 0.24
    for offset, method in enumerate(method_order):
        values = []
        for part_size in part_order:
            subset = comparison[(comparison['method'] == method) & (comparison['part_size'] == part_size)]
            values.append(subset[metric].mean())
        ax.bar(positions + (offset - 1) * width, values, width=width, label=method, color=colors[method])
    ax.set_title(title + '\n' + subtitle, fontsize=10)
    ax.set_xticks(positions)
    ax.set_xticklabels(part_order)
    ax.grid(axis='y', alpha=0.25)
    if metric == 'pred_to_gt_area_ratio':
        ax.axhline(1.0, color='black', linestyle='--', linewidth=0.8)
axes[0].legend(loc='upper left', fontsize=8)
fig.tight_layout()
chart_path = FIGURE_DIR / 'hybrid_localization_metric_comparison.png'
fig.savefig(chart_path, bbox_inches='tight')
print('saved:', chart_path)
plt.show()


## 7. Per-Case Deltas

This table shows where the hybrid improves or worsens relative to original FYS-TDM. It is useful for choosing representative cases before running expensive hybrid editing.


In [ ]:
fys_wide = comparison[comparison['method'] == 'FYS-TDM'].set_index('run_uid')
hybrid_wide = comparison[comparison['method'] == 'Attention-gated TDM'].set_index('run_uid')
attention_wide = comparison[comparison['method'] == 'FLUX edit+part attention'].set_index('run_uid')

delta = hybrid_wide[['case_uid', 'seed', 'part_size', 'part', 'edit', 'binary_iou', 'soft_ap', 'pred_to_gt_area_ratio']].copy()
delta = delta.rename(columns={
    'binary_iou': 'hybrid_iou',
    'soft_ap': 'hybrid_ap',
    'pred_to_gt_area_ratio': 'hybrid_area_ratio',
})
delta['fys_iou'] = fys_wide['binary_iou']
delta['fys_ap'] = fys_wide['soft_ap']
delta['fys_area_ratio'] = fys_wide['pred_to_gt_area_ratio']
delta['attention_iou'] = attention_wide['binary_iou']
delta['attention_ap'] = attention_wide['soft_ap']
delta['attention_area_ratio'] = attention_wide['pred_to_gt_area_ratio']
delta['hybrid_minus_fys_iou'] = delta['hybrid_iou'] - delta['fys_iou']
delta['hybrid_minus_fys_ap'] = delta['hybrid_ap'] - delta['fys_ap']
delta['area_ratio_reduction_vs_fys'] = delta['fys_area_ratio'] - delta['hybrid_area_ratio']
delta = delta.reset_index().sort_values(['part_size', 'hybrid_minus_fys_iou'], ascending=[True, False])

delta_path = RESULT_DIR / 'hybrid_vs_fys_delta.csv'
delta.to_csv(delta_path, index=False)
print('saved:', delta_path)

display(delta[['run_uid', 'part_size', 'part', 'edit', 'fys_iou', 'attention_iou', 'hybrid_iou', 'hybrid_minus_fys_iou', 'fys_area_ratio', 'attention_area_ratio', 'hybrid_area_ratio']].round(3))


## 8. Visualize Representative Masks

Each row shows one case only; repeated seeds from the same source image are intentionally removed from this figure. The goal is to compare localization behavior, not seed-level variance.

Columns show source image, GT part mask, original FYS-TDM, FLUX edit+part token attention, hybrid soft map, and hybrid binary mask. This is the most direct check for whether attention-gated TDM shrinks over-localized TDM in the intended direction.


In [ ]:
def to_uint8(image: np.ndarray) -> np.ndarray:
    if image.dtype == np.uint8:
        return image
    return np.clip(image * 255, 0, 255).astype(np.uint8)


def load_rgb(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    image = Image.open(REPO_ROOT / path).convert('RGB')
    if size is not None and image.size != size:
        image = image.resize(size, Image.Resampling.BICUBIC)
    return np.asarray(image).astype(np.float32) / 255.0


def load_mask_image(path: str, size: tuple[int, int]) -> np.ndarray:
    image = Image.open(REPO_ROOT / path).convert('L').resize(size, Image.Resampling.NEAREST)
    return np.asarray(image) > 0


def resize_score_to_image(score: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    score_norm = minmax_normalize(score)
    score_image = Image.fromarray((score_norm * 255).astype(np.uint8), mode='L').resize(size, Image.Resampling.BICUBIC)
    return np.asarray(score_image).astype(np.float32) / 255.0


def overlay_score(image: np.ndarray, score: np.ndarray, color=(255, 0, 255), alpha=0.45) -> Image.Image:
    base = Image.fromarray(to_uint8(image)).convert('RGBA')
    score_uint8 = (np.clip(score, 0, 1) * 255).astype(np.uint8)
    color_layer = Image.new('RGBA', base.size, (*color, 0))
    color_layer.putalpha(Image.fromarray((score_uint8 * alpha).astype(np.uint8)))
    return Image.alpha_composite(base, color_layer).convert('RGB')


def build_mask_panel(row: pd.Series, thumb=(170, 170), text_width=330) -> Image.Image:
    source = load_rgb(row['source_image'])
    image_size = Image.open(REPO_ROOT / row['source_image']).size
    gt = load_mask_image(row['gt_mask'], image_size)
    tdm = resize_score_to_image(load_soft_map(row['fys_tdm_path']), image_size)
    attention_score = resize_score_to_image(load_soft_map(row['attention_path']), image_size)
    hybrid_score = resize_score_to_image(load_soft_map(row['hybrid_smoothed_path']), image_size)
    hybrid_binary = resize_score_to_image(load_soft_map(row['hybrid_binary_path']), image_size)

    panels = [
        Image.fromarray(to_uint8(source)),
        overlay_score(source, gt.astype(np.float32), color=(255, 0, 0), alpha=0.5),
        overlay_score(source, tdm, color=(255, 0, 255), alpha=0.45),
        overlay_score(source, attention_score, color=(0, 150, 255), alpha=0.45),
        overlay_score(source, hybrid_score, color=(0, 200, 80), alpha=0.5),
        overlay_score(source, hybrid_binary, color=(255, 220, 0), alpha=0.55),
    ]
    panels = [p.resize(thumb, Image.Resampling.BICUBIC) for p in panels]
    labels = ['source', 'GT part', 'FYS TDM', 'edit+part attn', 'hybrid soft', 'hybrid bin']

    font = ImageFont.load_default()
    height = thumb[1] + 36
    width = text_width + len(panels) * thumb[0]
    canvas = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(canvas)
    lines = [
        f"{row['run_uid']} | {row['part_size']} | {row['part']} -> {row['edit']}",
        f"IoU FYS={row['fys_iou']:.3f}, Attn={row['attention_iou']:.3f}, Hybrid={row['hybrid_iou']:.3f}",
        f"Area FYS={row['fys_area_ratio']:.2f}, Attn={row['attention_area_ratio']:.2f}, Hybrid={row['hybrid_area_ratio']:.2f}",
    ]
    y = 6
    for line in lines:
        draw.text((8, y), line, fill='black', font=font)
        y += 13
    x = text_width
    for label, panel in zip(labels, panels):
        draw.text((x + 4, 4), label, fill='black', font=font)
        canvas.paste(panel, (x, 22))
        x += thumb[0]
    return canvas

# Select high-signal examples, but keep at most one seed per case/image.
def append_unique_cases(target: list[str], candidates: list[str], max_items: int | None = None) -> None:
    seen_cases = set(delta[delta['run_uid'].isin(target)]['case_uid']) if target else set()
    for run_uid in candidates:
        case_uid = str(delta.loc[delta['run_uid'] == run_uid, 'case_uid'].iloc[0])
        if case_uid in seen_cases:
            continue
        target.append(run_uid)
        seen_cases.add(case_uid)
        if max_items is not None and len(target) >= max_items:
            break

representative_run_uids = []
append_unique_cases(
    representative_run_uids,
    delta.sort_values('hybrid_minus_fys_iou', ascending=False)['run_uid'].tolist(),
    max_items=3,
)
append_unique_cases(
    representative_run_uids,
    delta.sort_values('hybrid_minus_fys_iou', ascending=True)['run_uid'].tolist(),
    max_items=6,
)
append_unique_cases(
    representative_run_uids,
    delta[delta['part_size'] == 'small'].sort_values('area_ratio_reduction_vs_fys', ascending=False)['run_uid'].tolist(),
    max_items=8,
)

visual_rows = hybrid_metrics.merge(
    delta[['run_uid', 'fys_iou', 'attention_iou', 'hybrid_iou', 'fys_area_ratio', 'attention_area_ratio', 'hybrid_area_ratio']],
    on='run_uid',
)
visual_rows = visual_rows[visual_rows['run_uid'].isin(representative_run_uids)].set_index('run_uid').loc[representative_run_uids].reset_index()

if visual_rows['case_uid'].duplicated().any():
    duplicated = visual_rows.loc[visual_rows['case_uid'].duplicated(), 'case_uid'].tolist()
    raise AssertionError(f'Representative figure repeats case_uid values: {duplicated}')

print('representative cases:', visual_rows[['run_uid', 'case_uid', 'seed', 'part_size', 'part', 'edit']].to_dict('records'))

panels = [build_mask_panel(row) for _, row in visual_rows.iterrows()]
sheet_width = max(panel.width for panel in panels)
sheet_height = sum(panel.height for panel in panels)
sheet = Image.new('RGB', (sheet_width, sheet_height), 'white')
y = 0
for panel in panels:
    sheet.paste(panel, (0, y))
    y += panel.height

sheet_path = FIGURE_DIR / 'attention_gated_tdm_representative_masks.jpg'
sheet.save(sheet_path, quality=95)
print('saved:', sheet_path)
display(sheet)


## 9. Failure Case Analysis

These examples are intentionally selected failure or weak cases. They show why edit+part attention is useful but not yet a clean part-localization signal.

The common pattern is that the edit token can carry strong object-, category-, or material-level semantics. When that happens, the edit+part attention map may expand beyond the requested part. The hybrid mask then improves over raw TDM in some cases, but it can still inherit broad semantic activation from the edit token.

This motivates a stricter next step: use part-only attention, or a token-disentangled gate where the part token determines the spatial region and the edit token supplies the new semantics.


In [ ]:
failure_case_specs = [
    {
        'case_uid': 'real_0006',
        'name': 'Head -> alien',
        'failure_mode': 'The edit token changes object identity, so edit+part attention can spread over the full person instead of only the head.',
    },
    {
        'case_uid': 'real_0000',
        'name': 'Torso -> armored',
        'failure_mode': 'Armored is a body-level style/category attribute; attention can cover limbs and the robot body, not only the torso panel.',
    },
    {
        'case_uid': 'real_0004',
        'name': 'Carhood -> rusted',
        'failure_mode': 'Rusted is a material attribute that can apply to the whole car surface, so the attention gate may not isolate the hood cleanly.',
    },
]

failure_lookup = hybrid_metrics.merge(
    delta[['run_uid', 'fys_iou', 'attention_iou', 'hybrid_iou', 'fys_area_ratio', 'attention_area_ratio', 'hybrid_area_ratio']],
    on='run_uid',
)

failure_rows = []
for spec in failure_case_specs:
    subset = failure_lookup[failure_lookup['case_uid'] == spec['case_uid']].sort_values('seed')
    if subset.empty:
        print('missing failure case:', spec['case_uid'])
        continue
    row = subset.iloc[0].copy()
    row['failure_name'] = spec['name']
    row['failure_mode'] = spec['failure_mode']
    failure_rows.append(row)

failure_rows = pd.DataFrame(failure_rows)
if failure_rows.empty:
    raise ValueError('No failure cases were found in hybrid_metrics.')

display(failure_rows[[
    'run_uid', 'failure_name', 'part_size', 'part', 'edit',
    'fys_iou', 'attention_iou', 'hybrid_iou',
    'fys_area_ratio', 'attention_area_ratio', 'hybrid_area_ratio',
    'failure_mode',
]].round(3))

failure_panels = [build_mask_panel(row) for _, row in failure_rows.iterrows()]
failure_sheet_width = max(panel.width for panel in failure_panels)
failure_sheet_height = sum(panel.height for panel in failure_panels)
failure_sheet = Image.new('RGB', (failure_sheet_width, failure_sheet_height), 'white')
y = 0
for panel in failure_panels:
    failure_sheet.paste(panel, (0, y))
    y += panel.height

failure_sheet_path = FIGURE_DIR / 'attention_gated_tdm_failure_cases.jpg'
failure_sheet.save(failure_sheet_path, quality=95)
print('saved:', failure_sheet_path)
display(failure_sheet)

print('Interpretation:')
print('- Edit+part attention is not a pure part mask; strong edit tokens can activate the whole object.')
print('- Attention-gated TDM can reduce over-localized TDM, but it may still fail when the edit token is globally grounded.')
print('- A better next experiment is part-only attention or a token-disentangled gate: use the part token for where, and the edit token for what.')


## 10. Part-Only Attention Hypothesis

The failure cases above suggest a more specific hypothesis: the edit token often answers *what to generate*, while the part token should better answer *where to edit*.

Therefore, the next diagnostic should compare the current edit+part attention map against a stricter part-only attention map. If part-only attention is more localized on cases such as `head -> alien`, `torso -> armored`, or `carhood -> rusted`, it would support a token-disentangled design: use the part token for the spatial gate and the edit token only for the injected editing semantics.

This section is optional and will run only if `core/results/flux_part_attention_baseline/` is available.


In [ ]:
part_only_root = BASE_RESULTS_DIR / 'flux_part_attention_baseline'
part_only_metrics_path = RESULT_DIR / 'part_only_attention_metrics.csv'
expected_run_count = runs['run_uid'].nunique()
part_only_count = len(list(part_only_root.glob('*/seed_*/attention_proxy_smoothed.npy')))

print('expected part-only attention maps:', expected_run_count)
print('available part-only attention maps:', part_only_count)

if part_only_count < expected_run_count:
    print('\nPart-only attention results are not available yet. This is expected if the server run was skipped.')
    print('Run this on a GPU machine once HuggingFace model access is working:')
    print('python core/scripts/run_flux_attention_baseline.py \\')
    print('  --manifest core/data/partedit_subset/pilot_12_manifest.json \\')
    print('  --seeds 0,1,2 \\')
    print('  --token-mode part \\')
    print('  --execute')
else:
    part_only_attention = build_attention_metrics_from_root(fys, part_only_root)
    part_only_attention.to_csv(part_only_metrics_path, index=False)
    print('saved:', part_only_metrics_path)

    part_only_long = part_only_attention[[
        'run_uid', 'case_uid', 'seed', 'part_size', 'part', 'edit',
        'binary_iou', 'soft_ap', 'pred_to_gt_area_ratio', 'soft_inside_gt_mass',
    ]].copy()
    part_only_long['method'] = 'FLUX part-only attention'

    edit_part_long = comparison[comparison['method'] == 'FLUX edit+part attention'][[
        'run_uid', 'case_uid', 'seed', 'part_size', 'part', 'edit',
        'binary_iou', 'soft_ap', 'pred_to_gt_area_ratio', 'soft_inside_gt_mass', 'method',
    ]]

    part_comparison = pd.concat([edit_part_long, part_only_long], ignore_index=True)
    part_summary = (
        part_comparison
        .groupby(['method', 'part_size'], as_index=False)
        .agg(
            n_runs=('run_uid', 'count'),
            binary_iou=('binary_iou', 'mean'),
            soft_ap=('soft_ap', 'mean'),
            area_ratio=('pred_to_gt_area_ratio', 'mean'),
            inside_mass=('soft_inside_gt_mass', 'mean'),
        )
    )
    display(part_summary.round(3))

    # Visualize part-only attention on the same failure cases, one seed per case.
    part_only_lookup = part_only_attention.set_index('run_uid')
    visual_failure_rows = []
    for _, base_row in failure_rows.iterrows():
        run_uid = base_row['run_uid']
        if run_uid not in part_only_lookup.index:
            continue
        part_row = part_only_lookup.loc[run_uid]
        visual_failure_rows.append({
            'run_uid': run_uid,
            'source_image': base_row['source_image'],
            'gt_mask': base_row['gt_mask'],
            'edit_part_attention_path': base_row['attention_path'],
            'part_only_attention_path': part_row['attention_smoothed_path'],
            'part': base_row['part'],
            'edit': base_row['edit'],
            'edit_part_iou': base_row['attention_iou'],
            'part_only_iou': part_row['binary_iou'],
            'edit_part_area': base_row['attention_area_ratio'],
            'part_only_area': part_row['pred_to_gt_area_ratio'],
        })

    def build_part_only_panel(row: dict, thumb=(180, 180), text_width=360) -> Image.Image:
        source = load_rgb(row['source_image'])
        image_size = Image.open(REPO_ROOT / row['source_image']).size
        gt = load_mask_image(row['gt_mask'], image_size)
        edit_part_score = resize_score_to_image(load_soft_map(row['edit_part_attention_path']), image_size)
        part_only_score = resize_score_to_image(load_soft_map(row['part_only_attention_path']), image_size)
        panels = [
            Image.fromarray(to_uint8(source)),
            overlay_score(source, gt.astype(np.float32), color=(255, 0, 0), alpha=0.5),
            overlay_score(source, edit_part_score, color=(0, 150, 255), alpha=0.45),
            overlay_score(source, part_only_score, color=(0, 200, 80), alpha=0.5),
        ]
        panels = [panel.resize(thumb, Image.Resampling.BICUBIC) for panel in panels]
        labels = ['source', 'GT part', 'edit+part attn', 'part-only attn']
        font = ImageFont.load_default()
        canvas = Image.new('RGB', (text_width + len(panels) * thumb[0], thumb[1] + 36), 'white')
        draw = ImageDraw.Draw(canvas)
        lines = [
            f"{row['run_uid']} | {row['part']} -> {row['edit']}",
            f"IoU edit+part={row['edit_part_iou']:.3f}, part-only={row['part_only_iou']:.3f}",
            f"Area edit+part={row['edit_part_area']:.2f}, part-only={row['part_only_area']:.2f}",
        ]
        y = 6
        for line in lines:
            draw.text((8, y), line, fill='black', font=font)
            y += 13
        x = text_width
        for label, panel in zip(labels, panels):
            draw.text((x + 4, 4), label, fill='black', font=font)
            canvas.paste(panel, (x, 22))
            x += thumb[0]
        return canvas

    if visual_failure_rows:
        part_panels = [build_part_only_panel(row) for row in visual_failure_rows]
        sheet = Image.new('RGB', (max(panel.width for panel in part_panels), sum(panel.height for panel in part_panels)), 'white')
        y = 0
        for panel in part_panels:
            sheet.paste(panel, (0, y))
            y += panel.height
        part_only_sheet_path = FIGURE_DIR / 'part_only_attention_failure_case_check.jpg'
        sheet.save(part_only_sheet_path, quality=95)
        print('saved:', part_only_sheet_path)
        display(sheet)


## 11. Output Files

Generated files:

- `core/results/attention_gated_tdm/hybrid_localization_metrics.csv`
- `core/results/attention_gated_tdm/hybrid_localization_comparison_long.csv`
- `core/results/attention_gated_tdm/hybrid_localization_summary.csv`
- `core/results/attention_gated_tdm/hybrid_vs_fys_delta.csv`
- `core/results/attention_gated_tdm/hybrid_masks/<case_uid>/seed_<seed>/attention_gated_tdm_*.npy`
- `core/results/attention_gated_tdm/figures/hybrid_localization_metric_comparison.png`
- `core/results/attention_gated_tdm/figures/attention_gated_tdm_representative_masks.jpg`
- `core/results/attention_gated_tdm/figures/attention_gated_tdm_failure_cases.jpg`

These files are diagnostic inputs for the next stage. They do not replace the existing controlled-revision result tables.
